In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [2]:
credentials_location = '/home/moha_/.google/credentials/google_credentials.json'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-hadoop3-2.2.5.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [3]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

26/03/04 23:40:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [5]:
df_green = spark.read.parquet('gs://zoomcamp_spark_module6/pq/green/*/*')

In [6]:
df_green.count()

2304517

In [7]:
# What: Check the Spark Context's loaded JARs
# Why: To confirm the GCS connector is officially part of the session
print(spark.sparkContext._gateway.jvm.java.lang.System.getProperty("java.class.path"))

# What: List files in your GCS bucket using Spark
# Why: If this works, your JAR and credentials are perfect!
# Replace 'your-bucket-name' with your actual bucket name
bucket = "zoomcamp_spark_module6"
try:
    df_test = spark.read.parquet(f"gs://{bucket}/pq/")
    print("\n✅ Success! Spark can read from your GCS bucket.")
    df_test.show(5)
except Exception as e:
    print(f"\n❌ Still a hiccup: {e}")

/home/moha_/spark-3.5.0-bin-hadoop3/conf/:/home/moha_/spark-3.5.0-bin-hadoop3/jars/spark-sketch_2.12-3.5.0.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/spark-tags_2.12-3.5.0.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/commons-crypto-1.1.0.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/jdo-api-3.0.1.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/spark-mllib-local_2.12-3.5.0.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/spark-mesos_2.12-3.5.0.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/log4j-1.2-api-2.20.0.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/kubernetes-model-policy-6.7.2.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/hive-beeline-2.3.9.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/netty-resolver-4.1.96.Final.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/blas-3.0.3.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/zookeeper-3.6.3.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/jakarta.ws.rs-api-2.1.6.jar:/home/moha_/spark-3.5.0-bin-hadoop3/jars/zjsonpatch-0.3.0.jar:/home/moha_/spark-3.5.0-bin-h

In [9]:
# What: Define the paths to your cleaned GCS folders
# Why: These match the exact structure we see in your Bucket browser
green_path = "gs://zoomcamp_spark_module6/pq/green/"
yellow_path = "gs://zoomcamp_spark_module6/pq/yellow/"

try:
    print("--- Reading Green Taxi Data from GCS ---")
    df_green = spark.read.parquet(green_path)
    df_green.show(5)
    
    print("\n--- Reading Yellow Taxi Data from GCS ---")
    df_yellow = spark.read.parquet(yellow_path)
    df_yellow.show(5)
    
    print("\n✅ Success! Spark is fully integrated with your GCS Data Lake.")
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("Hint: If you still get 'Inferred Schema' errors, run 'gsutil ls -R gs://zoomcamp_spark_module6/pq/green/' to ensure files exist inside.")

--- Reading Green Taxi Data from GCS ---

❌ Error: [UNABLE_TO_INFER_SCHEMA] Unable to infer schema for Parquet. It must be specified manually.
Hint: If you still get 'Inferred Schema' errors, run 'gsutil ls -R gs://zoomcamp_spark_module6/pq/green/' to ensure files exist inside.


In [10]:
# What: Use a wildcard (**) to find parquet files in subfolders
# Why: If your data is in green/2019/01/data.parquet, the base path green/ might fail
green_wildcard_path = "gs://zoomcamp_spark_module6/pq/green/*/*/*.parquet"

try:
    print("Searching for files with wildcard...")
    df_green = spark.read.parquet(green_wildcard_path)
    df_green.show(5)
    print("✅ Success!")
except Exception as e:
    print(f"❌ Wildcard failed too: {e}")

Searching for files with wildcard...


+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2020-01-23 13:10:15|  2020-01-23 13:38:16|                 N|         1|          74|         130|              1|        12.77|       36.0|  0.0|    0.